# 02 - Realism checks

PRD section 3.7, flow 3. Five checks, one report (`results/realism_report.json` in Drive):

1. Automatic filter of bad images, with the reason for every rejection (Grounding-DINO as the teacher, course 2263)
2. FID between synthetic and real images (`torchmetrics`, 2048-dim Inception features, course 2264)
3. Inception Score of the synthetic images (course 2264)
4. Coverage table: images per object type, size and condition
5. Human eye test: 20 real + 20 synthetic, shuffled

The filter result is written into each image's `meta/*.json`; rejected images stay in `synthetic/`, marked as rejected.

In [ ]:
%pip install -q transformers accelerate torchmetrics torch-fidelity

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch

import os, subprocess, sys, yaml
REPO = "/content/genai-shelter-validator"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))
sys.path.insert(0, f"{REPO}/Code")

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
os.makedirs(ROOT, exist_ok=True)
print("Config loaded, Drive ready at", ROOT)

In [ ]:
import json, random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
import common as C

device = C.device
val = cfg["validation"]
table = C.load_objects(REPO, cfg)
classes = table["name"].tolist()
meta = C.read_meta(ROOT)
assert meta, "No synthetic images in Drive. Run 01_generate first."
real = C.split_real(C.load_real(ROOT, classes), cfg, ROOT)
os.makedirs(f"{ROOT}/results", exist_ok=True)
print(f"{len(meta)} synthetic images, {len(real)} real images")

## 1. Automatic filter of bad images (with the reason for each rejection)

* **Destroyed image**: almost no pixel variation (flat, black or white).
* **Object not visible** (method A): Grounding-DINO is asked for the object (and, for a lookalike, also for its genuine
  twin) and must find something that overlaps the saved box (`dino_iou`). Method B only gets the first check: a zero-shot
  detector cannot be trusted on purpose-darkened or blurred images, so "objects did not move" stays a manual check.

Thresholds are in `config.yaml` (`validation`). Look at the rejected images below and adjust them if the filter is too
strict or too loose; set `RECHECK = True` to run the filter again on everything.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

RECHECK = False
dino_processor = AutoProcessor.from_pretrained(val["dino_model"])
dino = AutoModelForZeroShotObjectDetection.from_pretrained(val["dino_model"]).to(device).eval()


def dino_boxes(img, text, threshold):
    inputs = dino_processor(images=img, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = dino(**inputs)
    res = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=threshold, text_threshold=threshold, target_sizes=[img.size[::-1]])[0]
    return res["boxes"].detach().cpu().numpy()


def check_image(m):
    img = Image.open(f"{ROOT}/synthetic/{m['file']}").convert("RGB")
    pixels = np.asarray(img, dtype=float)
    if not np.isfinite(pixels).all() or pixels.std() < val["min_std"]:
        return False, "destroyed image (almost no pixel variation)"
    if m["method"] != "add_objects":
        return True, ""
    name = m["boxes"][0]["name"]
    parent = table.loc[table["name"] == name, "lookalike_of"].iloc[0] or name
    asked = list(dict.fromkeys([C.phrase(name), C.phrase(parent)]))
    best = max([C.iou(b, m["boxes"][0]["xyxy"]) for b in dino_boxes(img, ". ".join(asked) + ".", val["dino_box_threshold"])],
               default=0.0)
    if best < val["dino_iou"]:
        return False, f"object not visible: {asked} not found at the saved box (best IoU {best:.2f})"
    return True, ""


for m in tqdm(meta):
    if m["filter"] is None or RECHECK:
        passed, reason = check_image(m)
        m["filter"] = {"passed": passed, "reason": reason}
        json.dump(m, open(f"{ROOT}/synthetic/meta/{m['id']}.json", "w"), indent=1)

log = pd.DataFrame([{"id": m["id"], "method": m["method"], "object": m["boxes"][0]["name"] if m["method"] == "add_objects" else "(real boxes)", "condition": m["condition"],
                     "passed": m["filter"]["passed"], "reason": m["filter"]["reason"]} for m in meta])
log.to_csv(f"{ROOT}/results/filter_log.csv", index=False)
print(f"passed {int(log['passed'].sum())} of {len(log)}")
display(log[~log["passed"]].groupby(["method", "reason"]).size().rename("rejected").reset_index().head(20))

## 2-3. FID and Inception Score

`FrechetInceptionDistance(feature=2048)` and `InceptionScore` from `torchmetrics`. Both use ImageNet Inception features, so
IS says little about shelters (PRD 14); we report it without a target. FID needs hundreds of images per side to be stable:
with about 100 real images the number is biased upwards. To make it readable, the **real-vs-real** row splits the real
images in two halves and shows the floor that no generator can beat with this much real data.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

rng = random.Random(cfg["seed"])
passed_meta = [m for m in meta if m["filter"]["passed"]]
syn_paths = {"A": [], "B": []}
for m in passed_meta:
    syn_paths["A" if m["method"] == "add_objects" else "B"].append(f"{ROOT}/synthetic/{m['file']}")
real_paths = [r["path"] for r in real]
MAX_IMAGES = 500


def batch(paths):
    arrays = [np.asarray(Image.open(p).convert("RGB").resize((299, 299), Image.BICUBIC)) for p in paths]
    return torch.from_numpy(np.stack(arrays)).permute(0, 3, 1, 2).to(device)    # uint8, as torchmetrics expects


def sample(paths):
    return rng.sample(paths, min(len(paths), MAX_IMAGES))


def fid_between(real_side, fake_side):
    metric = FrechetInceptionDistance(feature=2048).to(device)
    for paths, is_real in [(real_side, True), (fake_side, False)]:
        for i in range(0, len(paths), 32):
            metric.update(batch(paths[i:i + 32]), real=is_real)
    return float(metric.compute().detach().cpu())


def inception_score(paths):
    metric = InceptionScore().to(device)
    for i in range(0, len(paths), 32):
        metric.update(batch(paths[i:i + 32]))
    mean, std = metric.compute()
    return float(mean.detach().cpu()), float(std.detach().cpu())


half = len(real_paths) // 2
shuffled = rng.sample(real_paths, len(real_paths))
fid = {"real_vs_real (floor)": fid_between(shuffled[:half], shuffled[half:2 * half]),
       "A_vs_real": fid_between(real_paths, sample(syn_paths["A"])) if syn_paths["A"] else None,
       "B_vs_real": fid_between(real_paths, sample(syn_paths["B"])) if syn_paths["B"] else None,
       "A+B_vs_real": fid_between(real_paths, sample(syn_paths["A"] + syn_paths["B"]))}
inc = {"synthetic": inception_score(sample(syn_paths["A"] + syn_paths["B"])), "real": inception_score(sample(real_paths))}
print("FID (lower is better):", {k: None if v is None else round(v, 1) for k, v in fid.items()})
print("Inception Score mean/std:", {k: tuple(round(x, 2) for x in v) for k, v in inc.items()})

## 4. Coverage table: images per object type, size and condition

Counts only images that passed the filter. A class with fewer than `min_per_class` passed images is marked red (PRD 3.7).
Size classes follow COCO at 512 px: small < 32x32, medium < 96x96, large otherwise.

In [ ]:
rows = []
for m in passed_meta:
    for b in m["boxes"]:
        x0, y0, x1, y1 = b["xyxy"]
        area = (x1 - x0) * (y1 - y0)
        rows.append({"object": b["name"], "image": m["id"], "method": "A" if m["method"] == "add_objects" else "B",
                     "size": "small" if area < 32 ** 2 else "medium" if area < 96 ** 2 else "large",
                     "condition": m["condition"]})
boxes = pd.DataFrame(rows)
cover = pd.DataFrame(index=pd.Index(classes, name="object"))
cover["role"] = table["role"].values
cover["images_A"] = boxes[boxes["method"] == "A"].groupby("object")["image"].nunique().reindex(classes).fillna(0).astype(int).values
cover["images_B"] = boxes[boxes["method"] == "B"].groupby("object")["image"].nunique().reindex(classes).fillna(0).astype(int).values
cover["images_total"] = cover["images_A"] + cover["images_B"]
for size in ["small", "medium", "large"]:
    cover[size] = boxes[boxes["size"] == size].groupby("object").size().reindex(classes).fillna(0).astype(int).values
cover.to_csv(f"{ROOT}/results/coverage_table.csv")
display(cover.style.apply(lambda col: ["background-color: #f4b6b6" if v < val["min_per_class"] else "" for v in col],
                          subset=["images_total"]))
display(pd.crosstab(boxes["object"], boxes["condition"]))
enough = bool((cover["images_total"] >= val["min_per_class"]).all())
print("Every class has at least", val["min_per_class"], "passed images:", enough)

## 5. Human eye test: 20 real + 20 synthetic, shuffled

The grid is saved once (`eye_test.png`, with a secret `eye_test_key.json`). Everyone looks at it **without** opening the key,
the team agrees which numbers look real, and enters them below. 50% accuracy means the team cannot tell real from synthetic.

In [ ]:
import matplotlib.pyplot as plt

grid_path, key_path = f"{ROOT}/results/eye_test.png", f"{ROOT}/results/eye_test_key.json"
if not os.path.exists(key_path):
    items = [(p, "real") for p in rng.sample(real_paths, min(20, len(real_paths)))] + \
            [(p, "synthetic") for p in rng.sample(syn_paths["A"] + syn_paths["B"], 20)]
    rng.shuffle(items)
    fig, axes = plt.subplots(5, 8, figsize=(24, 15))
    for ax in axes.flat:
        ax.axis("off")
    for n, (ax, (path, _)) in enumerate(zip(axes.flat, items), start=1):
        ax.imshow(Image.open(path).convert("RGB").resize((256, 256)))
        ax.set_title(str(n), fontsize=16)
    plt.tight_layout()
    plt.savefig(grid_path, dpi=100)
    plt.close(fig)
    json.dump({str(n): kind for n, (_, kind) in enumerate(items, start=1)}, open(key_path, "w"))
display(Image.open(grid_path))

In [ ]:
GUESSED_REAL = []   # numbers 1-40 the team thinks are REAL, e.g. [1, 4, 5, ...]. Leave empty until the team has decided.

key = json.load(open(key_path))
if GUESSED_REAL:
    correct = sum((key[str(n)] == "real") == (n in GUESSED_REAL) for n in range(1, len(key) + 1))
    eye = {"images": len(key), "correct": correct, "accuracy": correct / len(key)}
    print(f"Team accuracy: {eye['accuracy']:.0%} ({correct} of {len(key)}); 50% = cannot tell them apart")
else:
    eye = {"status": "pending"}
    print("Eye test not done yet: look at eye_test.png and fill GUESSED_REAL.")

## 6. Realism report

In [ ]:
report = {
    "filter": {"images": len(log), "passed": int(log["passed"].sum()),
               "rejected_by_reason": log[~log["passed"]]["reason"].str.split(":").str[0].value_counts().to_dict()},
    "fid": fid,
    "inception_score": {k: {"mean": v[0], "std": v[1]} for k, v in inc.items()},
    "coverage": {"min_per_class": val["min_per_class"], "all_classes_enough": enough,
                 "images_per_class": cover["images_total"].to_dict()},
    "eye_test": eye,
}
json.dump(report, open(f"{ROOT}/results/realism_report.json", "w"), indent=1)
print(json.dumps(report, indent=1))